# Phase 5 — Train / Test Split Notebook

**Project:** AI-Powered Customer Feedback Intelligence System  
**Goal:** Split the cleaned review corpus (393,579 rows) into 80% Training and 20% Testing sets using stratified sampling to prevent class skew and data leakage.

### Objectives:
1. Load `data/processed/reviews_cleaned.csv`
2. Define Features ($X$) and Target ($y$)
3. Perform `train_test_split` with `test_size=0.20`, `random_state=42`, `stratify=y`
4. Verify Stratification and Class Distribution
5. Explain Data Leakage Prevention Principles

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Load cleaned dataset
data_path = Path('../data/processed/reviews_cleaned.csv')
df = pd.read_csv(data_path)
df['clean_text'] = df['clean_text'].fillna('')

print(f'Cleaned Dataset Loaded: {len(df):,} rows')
display(df[['Score', 'Sentiment', 'clean_text']].head(3))

## 2. Define Features (X) and Target (y)
We use `clean_text` as our input feature feature matrix $X$ and `Sentiment` as target vector $y$.

In [ ]:
X = df['clean_text']
y = df['Sentiment']
print(f'X shape: {X.shape}, y shape: {y.shape}')

## 3. Perform Train / Test Split
We split into **80% Train** (314,863 samples) and **20% Test** (78,716 samples).
* `random_state=42`: Guarantees reproducible results across environments.
* `stratify=y`: Preserves exact 77.9% Positive / 14.5% Negative / 7.6% Neutral class ratios in both splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set size : {len(X_train):,} ({len(X_train)/len(df)*100:.2f}%)')
print(f'Testing set size  : {len(X_test):,} ({len(X_test)/len(df)*100:.2f}%)')
print(f'Sum Verification  : {len(X_train) + len(X_test) == len(df)}')

## 4. Stratification & Distribution Verification
Comparing sentiment class percentages across Full Dataset, Train Set, and Test Set.

In [ ]:
dist_df = pd.DataFrame({
    'Full Dataset %': (y.value_counts(normalize=True)*100).round(2),
    'Train Set %': (y_train.value_counts(normalize=True)*100).round(2),
    'Test Set %': (y_test.value_counts(normalize=True)*100).round(2)
})
display(dist_df)

## 5. Data Leakage Prevention Rule
> [!IMPORTANT]
> **CRITICAL NLP RULE**: TF-IDF vectorizers MUST NOT be fitted on the combined dataset or test dataset.
> * In Phase 6, we will run `fit_transform()` ONLY on `X_train` to learn vocabulary and Inverse Document Frequencies (IDF).
> * `X_test` will ONLY be transformed using `tfidf.transform(X_test)`.